# NTNU Metamaterials Workshop — Colab (self-paced)

Run this **after the live demo** (~1 h on T4: install + training). Not designed for real-time use in a 45 min talk.

End-to-end **synthetic balls** 4D X-ray reconstruction (no local install). Kelvin/FEM path: see [WORKSHOP.md](https://github.com/igrega348/NTNU_metamaterials_workshop/blob/main/docs/WORKSHOP.md).

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Repo: [NTNU_metamaterials_workshop](https://github.com/igrega348/NTNU_metamaterials_workshop) · Method: [neural_xray / PNAS 2025](https://www.pnas.org/doi/10.1073/pnas.2521089122)


## Common errors

**`KeyError: 'optimizers'`** when restarting velocity-field training — delete partial outputs then re-run training:

```bash
rm -rf /content/NTNU_metamaterials_workshop/neural_xray/outputs/balls/xray_vfield/vel_6
rm -rf /content/NTNU_metamaterials_workshop/neural_xray/outputs/balls/xray_vfield/vel_12
```


In [ ]:
# @title Install dependencies (~8 min)

%cd /content/
!pip install -q --upgrade pip
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

!curl -sL "https://github.com/igrega348/tiny-cuda-nn-wheels/releases/download/1.7.3/tinycudann-2.0.post75260124-cp312-cp312-linux_x86_64.whl" -o tinycudann.whl
!pip install -q tinycudann.whl --force-reinstall

!apt-get -qq install colmap imagemagick

import os
REPO = "/content/NTNU_metamaterials_workshop"
if os.path.isdir(REPO):
    !rm -rf {REPO}
!git clone --recurse-submodules -q https://github.com/igrega348/NTNU_metamaterials_workshop.git
%cd {REPO}/neural_xray/nerfstudio
!pip install -q -e .
%cd {REPO}/neural_xray/nerfstudio-xray/nerf-xray
!pip install -q -e .
%cd {REPO}/neural_xray/xray_projection_render
!pip install -q -e .

%load_ext tensorboard
print("Install complete. Project root:", REPO)


In [ ]:
# @title Generate synthetic data (~2 min)
%cd /content/NTNU_metamaterials_workshop/neural_xray/scripts
!./generate_and_animate.sh


In [ ]:
# @title Preview synthetic data (GIFs)
from IPython.display import HTML
import base64

def b64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

base = "/content/NTNU_metamaterials_workshop/neural_xray/data/synthetic/balls"
rot = b64(base + "/rotation_animation.gif")
defm = b64(base + "/deformation_animation.gif")
HTML('<h3>Rotation</h3><img src="data:image/gif;base64,' + rot + '" width="400"/>'
     + '<h3>Deformation</h3><img src="data:image/gif;base64,' + defm + '" width="400"/>')


In [ ]:
# @title Train synthetic pipeline (~30 min on T4)
%cd /content
!pip install -q colab-xterm
%load_ext colabxterm
%env TERM=xterm
from IPython.display import clear_output
import os

clear_output(wait=True)
data_ok = os.path.exists("/content/NTNU_metamaterials_workshop/neural_xray/data/synthetic/balls/transforms_00.json")
if data_ok:
    print("\033[1mPaste this into the terminal below:\033[0m\n")
    print("cd /content/NTNU_metamaterials_workshop/neural_xray && bash scripts/demo_synthetic.sh")
else:
    print("Missing synthetic data — run the data generation cell first.")


In [ ]:
# @title TensorBoard
%tensorboard --logdir /content/NTNU_metamaterials_workshop/neural_xray/outputs/balls/
